In [ ]:
""" Log
-12/27/25 Testing how does the Gk(t,iw_n) look like 
"""

In [ ]:
### Plotting the analytic solution G(t, U, iw_n) and G(t(U), U, iw_n)
import sys
sys.path.append("../")
import numpy as np
import matplotlib.pyplot as plt
import utilities as ut
import itertools as it
import pickle
from pydlr import dlr
import cvxpy as cp
import SCE as sce
import WCE as wce

In [ ]:
beta = 20.0
Ntau = 401
Ns = 8
Nw = 400
Mat_fre = (2*np.linspace(0, Nw-1, Nw)+1)*np.pi/beta
wn = np.append(-np.flip(Mat_fre),Mat_fre)   # w_n = (2n+1)*pi/beta

In [ ]:
### Saving and Loading ED and MBPT data 
t_values_ED = np.around(np.linspace(0,2,21), decimals=1)
U_values_ED = 1+0*t_values_ED
print("ED t is :\n", t_values_ED)

full_sigma = "False"
data_directory_ED = '/global/cfs/cdirs/m1027/Yuan/EDLib_data/1d_hubbard/U1_t_var/'
data_directory_saving = 'ED_data'
ut.Saving_data_ED(Ns, t_values_ED , U_values_ED, Ntau, data_directory_ED, data_directory_saving, full_sigma, beta)
Gij_tau_ED, Gij_iwn_ED, Gk_tau_ED, Gk_iwn_ED  = ut.load_data_training_G_ED(data_directory_saving, Ns, beta)

print(Gij_tau_ED.shape)
print("For Ns=%d, Gij_tau_ED has the dimensionality len(U_values)*Ntau*Ns*Ns=%d*%d*%d*%d" %(Ns, Gij_tau_ED.shape[0], Gij_tau_ED.shape[1], Gij_tau_ED.shape[2], Gij_tau_ED.shape[3]))
print("For Ns=%d, Gij_iwn_ED has the dimensionality len(U_values)*Nw*Ns*Ns=%d*%d*%d*%d" %(Ns, Gij_iwn_ED.shape[0], Gij_iwn_ED.shape[1], Gij_iwn_ED.shape[2], Gij_iwn_ED.shape[3]))
        

In [ ]:
#### Plotting Gk_iwn 

fig, axes = plt.subplots(2, int(Ns/2), figsize=(18, 10),sharex=True, sharey=True)
k = 0
for i in range(2):
    for j in range(int(Ns/2)):
        ax = axes[i, j]
        for n in range(wn.size):
            ax.plot(t_values_ED, Gk_iwn_ED[:, k, n].real, 'C0')
        ax.set_title(r'$Re[G_{%d}(i\omega_n,U=%.1f, t)]$'%(k,U_values_ED[0]), fontsize=25)
        ax.tick_params(labelsize=25)
        k = k+1

        if i == Ns - 1:
            ax.set_xlabel("t", fontsize=25)
            ax.set_xlim([0,2])
            ax.set_ylim([-0.5,1])
        if j == 0:
            ax.set_ylabel(r"$G_k(t)$", fontsize=25)

plt.tight_layout()
plt.show()


fig, axes = plt.subplots(2, int(Ns/2), figsize=(18, 10),sharex=True, sharey=True)
k = 0
for i in range(2):
    for j in range(int(Ns/2)):
        ax = axes[i, j]
        for n in range(wn.size):
            ax.plot(t_values_ED, Gk_iwn_ED[:, k, n].imag, 'C1')
        ax.set_title(r'$Im[G_{%d}(i\omega_n,U=%.1f, t)]$'%(k,U_values_ED[0]), fontsize=25)
        ax.tick_params(labelsize=25)
        k = k+1

        if i == Ns - 1:
            ax.set_xlabel("t", fontsize=25)
            ax.set_xlim([0,2])
            ax.set_ylim([-0.5,1])
        if j == 0:
            ax.set_ylabel(r"$G_k(t)$", fontsize=25)

plt.tight_layout()
plt.show()

In [ ]:
t_values_wce = np.linspace(0.5,5,51)
t_values_sce = np.linspace(0,0.1,51)
Gij_iwn_wce = np.zeros((len(t_values_wce), wn.size, Ns, Ns), dtype ='complex')
Gij_iwn_sce = np.zeros((len(t_values_sce), wn.size, Ns, Ns), dtype ='complex')
Gij_iwn_hf = np.zeros((len(t_values_wce), wn.size, Ns, Ns), dtype ='complex')

Gk_iwn_hf = np.zeros((len(t_values_wce), Ns, wn.size), dtype ='complex')
Gk_iwn_sce = np.zeros((len(t_values_sce), Ns, wn.size), dtype ='complex')
Gk_iwn_wce = np.zeros((len(t_values_wce), Ns, wn.size), dtype ='complex')
for t in range(t_values_sce.size):
    Gij_iwn_sce[t]= sce.Gij_sce_t(2, Ns, 1.0, wn, t_values_sce[t])
    Gk_iwn_sce[t]= wce.Gij_to_Gk(Gij_iwn_sce[t])
for t in range(t_values_wce.size):
    Gij_iwn_hf[t], _ = wce.HF_G0(-t_values_wce[t], Ns, wn)
    Gij_iwn_wce[t]= wce.Gij_wce_U(2, Ns, 1.0, wn, beta, -t_values_wce[t])

    Gk_iwn_hf[t]= wce.Gij_to_Gk(Gij_iwn_hf[t])
    Gk_iwn_wce[t]= wce.Gij_to_Gk(Gij_iwn_wce[t])

In [ ]:
#### Plotting Gk_iwn 

fig, axes = plt.subplots(2, int(Ns/2), figsize=(18, 10),sharex=True, sharey=True)
k = 0
for i in range(2):
    for j in range(int(Ns/2)):
        ax = axes[i, j]
        for n in range(wn.size):
            ax.plot(t_values_ED, Gk_iwn_ED[:, k, n].real, 'C0')
            ax.plot(t_values_wce, Gk_iwn_wce[:, k, n].real, 'C1--')
            ax.plot(t_values_sce, Gk_iwn_sce[:, k, n].real, 'k--')

        ax.plot(t_values_ED, Gk_iwn_ED[:, k, n].real, 'C0', label = 'ED')
        ax.plot(t_values_wce, Gk_iwn_wce[:, k, n].real, 'C1--', label = 'wce')
        ax.plot(t_values_sce, Gk_iwn_sce[:, k, n].real, 'k--', label = 'sce')
        ax.set_title(r'$Re[G_{%d}(i\omega_n,U=%.1f, t)]$'%(k,U_values_ED[0]), fontsize=25)
        ax.tick_params(labelsize=25)
        ax.legend(frameon = False)
        k = k+1

        if i == Ns - 1:
            ax.set_xlabel("t", fontsize=25)
            ax.set_xlim([0,2])
            ax.set_ylim([-0.5,1])
        if j == 0:
            ax.set_ylabel(r"$G_k(t)$", fontsize=25)

plt.tight_layout()
plt.show()


fig, axes = plt.subplots(2, int(Ns/2), figsize=(18, 10),sharex=True, sharey=True)
k = 0
for i in range(2):
    for j in range(int(Ns/2)):
        ax = axes[i, j]
        for n in range(wn.size):
            ax.plot(t_values_ED, Gk_iwn_ED[:, k, n].imag, 'C0')
            ax.plot(t_values_wce, Gk_iwn_wce[:, k, n].imag, 'C1--')
            ax.plot(t_values_sce, Gk_iwn_sce[:, k, n].imag, 'k--')
            
        ax.plot(t_values_ED, Gk_iwn_ED[:, k, n].imag, 'C0', label = 'ED')
        ax.plot(t_values_wce, Gk_iwn_wce[:, k, n].imag, 'C1--', label = 'wce')
        ax.plot(t_values_sce, Gk_iwn_sce[:, k, n].imag, 'k--', label = 'sce')
        ax.set_title(r'$Im[G_{%d}(i\omega_n,U=%.1f, t)]$'%(k,U_values_ED[0]), fontsize=25)
        ax.tick_params(labelsize=25)
        ax.legend(frameon = False)
        k = k+1

        if i == Ns - 1:
            ax.set_xlabel("t", fontsize=25)
            ax.set_xlim([0,2])
            ax.set_ylim([-0.5,1])
        if j == 0:
            ax.set_ylabel(r"$G_k(t)$", fontsize=25)

plt.tight_layout()
plt.show()
